In [85]:
import pandas as pd
import numpy as np
import warnings

In [86]:
warnings.filterwarnings('ignore')

In [87]:
df = pd.read_csv('../data/curated/visualisation_df.csv')
df

,id,region,regulated_dam,primary_purpose,primary_type,height,length,volume,surface,drainage,...,assessment,probability_of_failure,dam_repair_loss,damage_loss,business_interruption_loss,age,modification_count,years_from_modification,years_from_inspection,years_from_assessment
0,SOAD00072,Navaldia,Yes,Recreation,Earth,NaN,NaN,NaN,0.02364,2.66329,...,Satisfactory,0.1258,20.8,296.9,8.1,NaN,0,NaN,10.0,NaN
1,SOAD00380,Navaldia,No,NaN,Earth,2.713,NaN,NaN,NaN,NaN,...,Not Available,0.0757,930.5,727.5,NaN,98.0,0,98.0,7.0,98.0
2,SOAD00610,Navaldia,Yes,Recreation,Earth,NaN,NaN,NaN,NaN,NaN,...,Satisfactory,0.1375,355.5,427.3,8.6,NaN,0,NaN,NaN,NaN
3,SOAD00862,Navaldia,Yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Not Rated,0.1403,295.1,25.3,NaN,NaN,0,NaN,NaN,NaN
4,SOAD02091,Lyndrassia,Yes,Recreation,Earth,13.921,0.200,NaN,0.04728,NaN,...,Not Rated,0.0998,11.5,203.0,5.6,44.0,0,44.0,4.0,44.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20801,SOAD16536,Lyndrassia,No,Flood Risk Reduction,Rockfill,35.872,1.667,7411000.0,480.62681,84247.85257,...,Not Available,0.0837,820.9,510.6,NaN,47.0,0,47.0,2.0,47.0
20802,SOAD13145,Lyndrassia,No,Hydroelectric,Gravity,141.296,0.963,375000.0,405.40236,62491.43656,...,Not Available,0.0795,850.2,176.6,62.9,52.0,0,52.0,5.0,52.0
20803,SOAD12688,Navaldia,No,Flood Risk Reduction,Earth,37.905,6.213,7370000.0,177.79053,25620.84980,...,Not Available,0.0972,922.4,490.6,NaN,71.0,0,71.0,3.0,71.0
20804,SOAD02340,Navaldia,No,Flood Risk Reduction,Earth,46.431,4.133,6000000.0,998.93972,24920.40453,...,Not Available,0.0672,652.4,603.7,NaN,60.0,0,60.0,4.0,60.0


In [88]:
df = df.loc[df['primary_type'] == 'Earth']

In [89]:
df

,id,region,regulated_dam,primary_purpose,primary_type,height,length,volume,surface,drainage,...,assessment,probability_of_failure,dam_repair_loss,damage_loss,business_interruption_loss,age,modification_count,years_from_modification,years_from_inspection,years_from_assessment
0,SOAD00072,Navaldia,Yes,Recreation,Earth,NaN,NaN,NaN,0.02364,2.66329,...,Satisfactory,0.1258,20.8,296.9,8.1,NaN,0,NaN,10.0,NaN
1,SOAD00380,Navaldia,No,NaN,Earth,2.713,NaN,NaN,NaN,NaN,...,Not Available,0.0757,930.5,727.5,NaN,98.0,0,98.0,7.0,98.0
2,SOAD00610,Navaldia,Yes,Recreation,Earth,NaN,NaN,NaN,NaN,NaN,...,Satisfactory,0.1375,355.5,427.3,8.6,NaN,0,NaN,NaN,NaN
4,SOAD02091,Lyndrassia,Yes,Recreation,Earth,13.921,0.200,NaN,0.04728,NaN,...,Not Rated,0.0998,11.5,203.0,5.6,44.0,0,44.0,4.0,44.0
5,SOAD02227,Lyndrassia,Yes,Recreation,Earth,14.332,0.127,NaN,0.06107,NaN,...,Unsatisfactory,0.1283,11.5,733.7,8.2,27.0,0,27.0,27.0,27.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20798,SOAD11040,Navaldia,No,Hydroelectric,Earth,3.956,0.105,NaN,1499.77873,50184.37347,...,Satisfactory,0.0956,850.1,163.8,83.8,59.0,0,59.0,3.0,2.0
20799,SOAD12695,Navaldia,No,Hydroelectric,Earth,3.444,0.230,NaN,1539.68108,49808.84958,...,Satisfactory,0.0772,770.7,331.4,87.1,59.0,0,59.0,3.0,2.0
20803,SOAD12688,Navaldia,No,Flood Risk Reduction,Earth,37.905,6.213,7370000.0,177.79053,25620.84980,...,Not Available,0.0972,922.4,490.6,NaN,71.0,0,71.0,3.0,71.0
20804,SOAD02340,Navaldia,No,Flood Risk Reduction,Earth,46.431,4.133,6000000.0,998.93972,24920.40453,...,Not Available,0.0672,652.4,603.7,NaN,60.0,0,60.0,4.0,60.0


In [90]:
df['damage_loss'] = df['damage_loss'].fillna(0)
df['dam_repair_loss'] = df['dam_repair_loss'].fillna(0)
df['business_interruption_loss'] = df['business_interruption_loss'].fillna(0)

In [91]:
df['business_interruption_loss'].isna().sum()

0

In [92]:
df['total_loss'] = df['damage_loss'] + df['dam_repair_loss'] + df['business_interruption_loss']
df['expected_loss'] = df['probability_of_failure'] * (df['damage_loss'] + df['dam_repair_loss'] + df['business_interruption_loss'])

### hazard

In [93]:
def calculate_hazard_rating_factor(df, hazard):
    curr_hazard_mean = (df[df['hazard'] == hazard].describe()['total_loss'].loc['50%'] + df[df['hazard'] == hazard].describe()['total_loss'].loc['50%']) / 2
    low_hazard_mean = (df[df['hazard'] == 'Low'].describe()['total_loss'].loc['50%'] + df[df['hazard'] == 'Low'].describe()['total_loss'].loc['50%']) / 2
    hazard_rating_factor = curr_hazard_mean/low_hazard_mean
    return hazard_rating_factor

In [94]:
hazard_low_rf = calculate_hazard_rating_factor(df, 'Low')
hazard_high_rf = calculate_hazard_rating_factor(df, 'High')
hazard_significant_rf = calculate_hazard_rating_factor(df, 'Significant')
hazard_undetermined_rf = calculate_hazard_rating_factor(df, 'Undetermined')

In [95]:
# Define mapping dictionary
hazard_mapping = {
    'Low': hazard_low_rf,
    'High': hazard_high_rf,
    'Significant': hazard_significant_rf,
    'Undetermined': hazard_undetermined_rf
}

# Create a new column using map()
df['hazard_rating_factor'] = df['hazard'].map(hazard_mapping)


In [96]:
df['region'].value_counts()

Navaldia      8374
Lyndrassia    7920
Flumevale     3074
Name: region, dtype: int64

### Regulation

In [97]:
df['no_BI_loss'] = df['dam_repair_loss'] + df['damage_loss']

In [98]:
df['w'] = df['no_BI_loss'] / df['total_loss']
df['w'] = df['w'].fillna(1)
df['failure_rate'] = df['probability_of_failure'] * df['w']
def calculate_regulation_rating_factor(df, region):
    regulated_region_failure_rate = sum(df[(df['region'] == region) & (df['regulated_dam'] == 'Yes')]['failure_rate'])
    unregulated_region_failure_rate = sum(df[(df['region'] == region) & (df['regulated_dam'] == 'No')]['failure_rate'])
    regulated_rating_factor = unregulated_region_failure_rate / regulated_region_failure_rate 
    return regulated_rating_factor

In [99]:
# Define mapping dictionary
regulated_mapping = {
    'Navaldia': calculate_regulation_rating_factor(df, 'Navaldia'),
    'Lyndrassia': calculate_regulation_rating_factor(df, 'Lyndrassia'),
    'Flumevale': calculate_regulation_rating_factor(df, 'Flumevale')
}

# Create a new column using map()
df['regulated_rating_factor'] = df['region'].map(regulated_mapping)


In [100]:
df['region'].value_counts()

Navaldia      8374
Lyndrassia    7920
Flumevale     3074
Name: region, dtype: int64

### GDP

In [101]:
gdp_df = pd.read_excel('../data/raw/soaGDP.xlsx', sheet_name='2025 Nominal GDP')
pop_df = pd.read_csv('../data/raw/soaPOP.csv')

In [102]:
pop_df

,Year,Flumevale,Lyndrassia,Navaldia,Tarrodan
0,2019,"45,363,514","7,067,855","39,808,697","92,240,066"
1,2020,"45,502,051","7,097,789","40,175,188","92,775,028"
2,2021,"45,651,175","7,131,024","40,565,887","93,348,086"
3,2022,"45,599,000","7,157,446","40,953,108","93,709,554"
4,2023,"45,311,937","7,239,138","42,148,205","94,699,280"


In [103]:
gdp_df

,Year,Flumevale,Lyndrassia,Navaldia,Tarrodan
0,2025,4.671259e+06,534401.257379,3.779710e+06,8.985371e+06


In [104]:
for feature in gdp_df.columns:
    gdp_df[feature] = gdp_df[feature].replace(to_replace=',', value= '', regex=True).astype(int)
    pop_df[feature] = pop_df[feature].replace(to_replace=',', value= '', regex=True).astype(int)

In [105]:
gdp_pp_df['Flumevale']

0    1.057087
Name: Flumevale, dtype: float64

In [106]:
gdp_pp_df = pd.DataFrame({
    'Year': gdp_df['Year']
})
for feature in gdp_df.drop(columns=['Year']).columns:
    gdp_pp_df[feature] = gdp_df[feature].item()/pop_df[feature]

In [107]:
gdp_pp_df

,Year,Flumevale,Lyndrassia,Navaldia,Tarrodan
0,2025,0.102974,0.07561,0.094947,0.097413


In [108]:
for feature in gdp_pp_df.drop(columns=['Year', 'Tarrodan']):
    gdp_pp_df[feature] = gdp_pp_df[feature]/gdp_pp_df['Tarrodan']

In [109]:
gdp_pp_df['Flumevale']

0    1.057087
Name: Flumevale, dtype: float64

In [110]:
regulated_mapping = {
    'Navaldia': gdp_pp_df['Navaldia'].item(),
    'Lyndrassia': gdp_pp_df['Lyndrassia'].item(),
    'Flumevale': gdp_pp_df['Flumevale'].item()
}

# Create a new column using map()
df['gdp_rating_factor'] = df['region'].map(regulated_mapping)


### summary

In [111]:
df['total_rating_factor'] = df['hazard_rating_factor'] * df['regulated_rating_factor'] * df['gdp_rating_factor']

In [112]:
df[['hazard_rating_factor', 'regulated_rating_factor', 'gdp_rating_factor', 'total_rating_factor']].describe()

,hazard_rating_factor,regulated_rating_factor,gdp_rating_factor,total_rating_factor
count,19368.000000,19368.000000,19368.000000,19368.000000
mean,2.791123,0.858094,0.906591,1.826658
std,2.677251,0.531792,0.112047,2.205589
min,0.095631,0.073407,0.776181,0.053440
25%,1.000000,0.573325,0.776181,0.558811
50%,1.000000,0.573325,0.974685,1.136135
75%,3.970874,1.463749,0.974685,1.136135
max,7.224272,1.463749,1.057087,8.207746
